# ViZDoom State & Frame Verification

using data from:
`/Users/arianamondiri/fmri-gym/data/sub-01_20260909-151927/`

## 1. Vizdoom corridor only expose health for gamevariable(1035(frames),1) - other versions might expose more

Games in `fmri-gym` are run via pre-configured curricula. The command you ran was:

```bash
python fmri_play.py --subject sub-01 --dummy-trigger --curriculum configs/dbp_games/vizdoom__deadly_corridor.json
```

* **Controls:** Arrow keys move/turn, `Z`/`X` strafe, `SPACE` shoots.
* **Saving:** Completely automatic! The session directory `data/<subject>_<timestamp>/` is written on finish or `ESC`.

In [ ]:
# Setup paths & imports
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import gymnasium as gym
import vizdoom.gymnasium_wrapper

sys.path.append(os.path.abspath("../"))

session_dir = os.path.abspath("../data/sub-01_20260909-151927")
manifest_path = os.path.join(session_dir, "manifest.json")
npz_path = os.path.join(session_dir, "block-02_vizdoom_VizdoomDefendCenter-v1.npz")

with open(manifest_path, "r") as f:
    manifest = json.load(f)
data = np.load(npz_path, allow_pickle=True)

print("Loaded session data successfully.")
print(f"Total frames recorded in session: {len(data['actions'])}")

In [ ]:
session_time = data["session_time"]
rewards = data["rewards"]
gamevars = data["gamevariables"]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(session_time, np.cumsum(rewards), color="teal", lw=1.6)
axes[0].set_ylabel("Cumulative Reward")
axes[0].set_title(f"Logged Gameplay Timeline Across {len(session_time)} Frames")
axes[0].grid(True, alpha=0.3)

if gamevars.shape[1] > 0:
    axes[1].plot(session_time, gamevars[:, 0], color="crimson", lw=1.6, label="Ammo (var 0)")
    if gamevars.shape[1] > 1:
        axes[1].plot(session_time, gamevars[:, 1], color="orange", lw=1.6, label="Health (var 1)")
    axes[1].set_ylabel("Game Variables")
    axes[1].set_xlabel("Session Time (seconds since trigger)")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

plt.tight_layout()
plt.show()

# NPZ

In [ ]:
import os
import numpy as np

npz_path = os.path.abspath("../data/sub-01_20260909-151927/block-02_vizdoom_VizdoomDefendCenter-v1.npz")
data = np.load(npz_path, allow_pickle=True)

print("=== ALL KEYS IN THE .NPZ FILE ===\n")
for key in data.files:
    arr = data[key]
    if hasattr(arr, "shape"):
        print(f"• {key:15s} | shape: {str(arr.shape):14s} | dtype: {arr.dtype}")
    else:
        print(f"• {key:15s} | {type(arr)}")

print("\n=== SAMPLES ===")
print("Actions shape (buttons):", data["actions"].shape)
print("Actions sample (frame 0):", data["actions"][0])
print("Game variables shape   :", data["gamevariables"].shape)
print("Game variables sample  :", data["gamevariables"][:5].flatten())
print("States array content   :", data["states"][:5])

## get pixel frame at t=10

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import vizdoom.gymnasium_wrapper

# 1. Load your saved session data
manifest_path = "../data/sub-01_20260909-151927/manifest.json"
npz_path = "../data/sub-01_20260909-151927/block-02_vizdoom_VizdoomDefendCenter-v1.npz"

with open(manifest_path, "r") as f:
    manifest = json.load(f)
data = np.load(npz_path, allow_pickle=True)

# 2. Find the frame at 10.0s into gameplay and slice its actions
target_time = data["session_time"][0] + 10.0
target_frame_idx = int(np.searchsorted(data["session_time"], target_time))

target_ep_id = int(data["episode_id"][target_frame_idx])
ep_seed = int(data["episode_seeds"][target_ep_id])
ep_start_frame = int(np.where(data["episode_id"] == target_ep_id)[0][0])
actions_to_replay = data["actions"][ep_start_frame : target_frame_idx + 1]

print(f"Replaying {len(actions_to_replay)} actions in Episode {target_ep_id} (Seed: {ep_seed}) to reach frame #{target_frame_idx}...")

# 3. Boot the environment, reset with seed, and replay
game_phase = next(p for p in manifest["curriculum"] if p.get("type") == "game")
env_kwargs = game_phase.get("env_kwargs", {})

env = gym.make("VizdoomDefendCenter-v1", render_mode="rgb_array", **env_kwargs)
env.reset(seed=ep_seed)
for a in actions_to_replay:
    env.step(a)

# 4. Render and clean up
frame_10s = env.render()
env.close()

# 5. Display the reconstructed frame
plt.figure(figsize=(8, 5))
plt.imshow(frame_10s)
plt.title(f"Reconstructed Frame at t = 10.0s (Global Frame #{target_frame_idx})", fontsize=12, fontweight="bold")
plt.axis("off")
plt.show()

## what is gamevariables

In [ ]:
import numpy as np
import gymnasium as gym
import vizdoom.gymnasium_wrapper

# 1. Check what was logged in your .npz file (Instant!)
gamevars_logged = data["gamevariables"]
print("=== LOGGED GAME VARIABLES IN .NPZ ===")
print(f"Shape: {gamevars_logged.shape} ({len(data['session_time'])} frames, {gamevars_logged.shape[1]} variables per frame)")
print(f"Value at frame 0  (start)       : {gamevars_logged[0]}")
print(f"Value at frame target (t=10.0s) : {gamevars_logged[target_frame_idx]}")
print(f"Min / Max values during run     : Min={gamevars_logged.min()}, Max={gamevars_logged.max()}")

# 2. Query ViZDoom for the exact variable names
temp_env = gym.make("VizdoomDefendCenter-v1")
game = temp_env.unwrapped.game
var_names = [str(v).split(".")[-1] for v in game.get_available_game_variables()]
temp_env.close()

print("\n=== VIZDOOM SCENARIO DEFINITION ===")
print(f"Variable names configured for this game: {var_names}")
for name, val in zip(var_names, gamevars_logged[target_frame_idx]):
    print(f" • {name}: {val}")


In [ ]:
import gymnasium as gym, vizdoom.gymnasium_wrapper
env = gym.make("VizdoomDefendCenter-v1")
print(env.unwrapped.game.get_available_game_variables())
env.close()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 2.2))
ax.axis("off")

table_data = [
    ["Pixels (Screen Frames)", "NO", "YES (via Replay)"],
    ["Symbolic State (AMMO & HEALTH)", "YES", "barely"],
    ["Hardware RAM / Blobs", "NO", "NO"]
]

col_labels = ["State Representation", "Saved in .NPZ?", "Reconstructable?"]

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    colColours=["#2b5c8f"] * 3
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(color="white", weight="bold")
    elif row == 3:
        cell.set_facecolor("#ffebee")
    elif "YES" in cell.get_text().get_text():
        cell.set_facecolor("#e8f5e9")

plt.tight_layout()
plt.show()